In [329]:
import numpy as np
import pandas as pd

####################################### Step 1: Convert Degrees, Minutes, Seconds to Decimal Degrees #############################################
def dms_to_dd(d, m, s):
    """
    Converts Degrees, Minutes, Seconds (DMS) to Decimal Degrees (DD).
    """
    return d + (m / 60) + (s / 3600)

# Given Corner Coordinates in DMS (Degrees, Minutes, Seconds)
top_left_lat_dms = (30, 12, 46.71)  # N
top_left_lon_dms = (-93, 15, 1.78)  # W
bottom_right_lat_dms = (30, 12, 31.27)  # N
bottom_right_lon_dms = (-93, 14.59, 47.40)  # W

# Convert DMS to Decimal Degrees
top_left_lat = dms_to_dd(*top_left_lat_dms)
top_left_lon = dms_to_dd(*top_left_lon_dms)
bottom_right_lat = dms_to_dd(*bottom_right_lat_dms)
bottom_right_lon = dms_to_dd(*bottom_right_lon_dms)

avg_lat = (top_left_lat + bottom_right_lat) / 2

####################################### Step 2: Define Grid Properties #############################################
grid_size_m = 50  # Grid cell size (50m x 50m)
earth_radius_m = 6371000  # Approximate Earth radius in meters

# Approximate meters per degree
meters_per_deg_lat = 111320  # Latitude conversion (approximately constant worldwide): 1∘latitude≈111320meters
meters_per_deg_lon = 111320 * np.cos(np.radians(top_left_lat))  # Longitude conversion (depends on latitude): 1∘longitude≈111320×cos(latitude)

# Compute number of rows and columns
num_rows = int(abs(top_left_lat - bottom_right_lat) * meters_per_deg_lat / grid_size_m) # H\50m
num_cols = int(abs(top_left_lon - bottom_right_lon) * meters_per_deg_lon / grid_size_m) # W\50m

# Verify the number of rows and columns
num_rows, num_cols, num_rows * num_cols

print(f"✅ Grid Dimensions: {num_rows} rows × {num_cols} columns (Total: {num_rows * num_cols} cells)")



✅ Grid Dimensions: 9 rows × 11 columns (Total: 99 cells)


In [331]:
####################################### Step 3: Generate Grid Centers #############################################
#each grid cell is positioned correctly before rotation
centers = []

for i in range(num_rows):
    for j in range(num_cols):
        lat = top_left_lat - ((i + 0.5) * (grid_size_m / meters_per_deg_lat))
        lon = top_left_lon + ((j + 0.5) * (grid_size_m / meters_per_deg_lon))
        centers.append((lat, lon))
        
# Convert to DataFrame
df_centers = pd.DataFrame(centers, columns=["X_Center", "Y_Center"]) # Changed order to (lat, lon)

# ✅ Fixing Numbering Order (Top-left → Down → Right)
df_centers = df_centers.sort_values(by=["X_Center", "Y_Center"], ascending=[True, True]).reset_index(drop=True)

# Add numbering for each cell (row-wise order: left to right, then move down)
df_centers.insert(0, "Cell Number", range(1, len(df_centers) + 1))

# # Save raw grid data
# df_centers.to_csv("Grid_Centers4.csv", index=False)
# print("✅ Grid Centers saved as 'Grid_Centers4.csv'")


In [333]:
# Display first few rows to select points manually
print(df_centers.head(99))  # Adjust the number to display more points


    Cell Number   X_Center   Y_Center
0             1  30.209157 -92.749246
1             2  30.209157 -92.748726
2             3  30.209157 -92.748206
3             4  30.209157 -92.747686
4             5  30.209157 -92.747167
..          ...        ...        ...
94           95  30.212750 -92.746127
95           96  30.212750 -92.745607
96           97  30.212750 -92.745088
97           98  30.212750 -92.744568
98           99  30.212750 -92.744048

[99 rows x 3 columns]


In [335]:
####################################### Step 5: Apply Rotation #############################################
theta_rad = np.radians(0)  # Convert degrees to radians

# Reference Point (Top-Left Corner) >> corner 1
x0, y0 = top_left_lat, top_left_lon

# Apply rotation to each grid center
rotated_centers = []
for lat, lon in centers:  
    # Convert original lat/lon to x/y distances from the reference point
    #translated the coordinates relative to reference point 
    #local coordinate system where (x0, y0) = (0,0)
    x = (lon - x0) * meters_per_deg_lon  # Convert longitude to meters
    y = (lat - y0) * meters_per_deg_lat  # Convert latitude to meters

    # Apply rotation around (x0, y0)
    x_rot = x * np.cos(theta_rad) - y * np.sin(theta_rad)
    y_rot = x * np.sin(theta_rad) + y * np.cos(theta_rad)

    # Convert back to lat/lon
    lon_rot = x0 + (x_rot / meters_per_deg_lon)
    lat_rot = y0 + (y_rot / meters_per_deg_lat)

    rotated_centers.append((lat_rot, lon_rot))

# Convert to DataFrame
df_rotated_centers = pd.DataFrame(rotated_centers, columns=["Y_Center", "X_Center"])

# Sort row-wise before numbering (left-to-right, then move down)
df_rotated_centers = df_rotated_centers.sort_values(by=["X_Center", "Y_Center"], ascending=[True, True]).reset_index(drop=True)

# Add numbering for each cell
df_rotated_centers.insert(0, "Cell Number", range(1, len(df_rotated_centers) + 1))

# Save results to a CSV file
df_rotated_centers.to_csv(r"C:\Users\abuoliem\Box\2.Second year2024\2.Spring2025\1.Research\2.satellite image analysis of pre- and post-disaster images\K.final files\reprojected coordinate\Location55.csv", index=False)

# Print confirmation message
print(f"File saved successfully as: {"Location44.csv"}")
print(df_rotated_centers.head())  # Show first few rows


File saved successfully as: Location44.csv
   Cell Number   Y_Center   X_Center
0            1  30.209157 -92.749246
1            2  30.209606 -92.749246
2            3  30.210055 -92.749246
3            4  30.210505 -92.749246
4            5  30.210954 -92.749246


In [21]:
# ####################################### Step 4: compute the rotation angle from coordinates manually  #############################################
# Manually define the correct top-left and bottom-left points
lat1, lon1 = 30.239832, -92.790859  # Top-left point
lat2, lon2 = 30.240281, -92.790859  # Bottom-left point

print(f"Manually Selected Top-Left: ({lat1}, {lon1})")
print(f"Manually Selected Bottom-Left: ({lat2}, {lon2})")

def compute_rotation_angle(lat1, lon1, lat2, lon2):
    """
    Computes the rotation angle relative to true north based on two grid points.
    """
    avg_lat = (lat1 + lat2) / 2
    meters_per_deg_lon = 111320 * np.cos(np.radians(avg_lat))

    delta_x = (lon2 - lon1) * meters_per_deg_lon  # Longitude difference in meters
    delta_y = (lat2 - lat1) * meters_per_deg_lat  # Latitude difference in meters

    angle_rad = np.arctan2(delta_x, delta_y)
    return np.degrees(angle_rad)

# Compute angle with manually set points
required_theta = compute_rotation_angle(lat1, lon1, lat2, lon2)
print(f"✅ Required Rotation Angle Before Applying Transformation: {required_theta:.6f} degrees")


Manually Selected Top-Left: (30.239832, -92.790859)
Manually Selected Bottom-Left: (30.240281, -92.790859)
✅ Required Rotation Angle Before Applying Transformation: 0.000000 degrees
